<div style="text-align:center;">
    <h1 >MERGE INTO</h1>
    <p>Strategy & Scenarios</p>
</div>




## 🗂️ Table of Contents

| # | Topic | Sub-scenarios |
|---|---|---|
| 1 | **What is MERGE INTO?** | UPSERT concept, why not overwrite |
| 2 | **Incremental Load Strategies** | Case A — Watermark/ID · Case B — CDC · Case C — File Metadata | 
| 3 | **Merge Without a Natural Key** | Composite Key · Full Row Hash · Surrogate Key · Window + Row Number · Fuzzy Matching |
| 4 | **Schema Evolution Handling** | Ignore · Auto-add · Fail · Approval · Store Separately · Version | 
| 5 | **File Format Handling** | Malformed CSV · Duplicate Headers · Malformed JSON · Mixed CSV+JSON |

---


###What is MERGE INTO?

MERGE INTO (also called UPSERT) is a Delta Lake operation that compares a source dataset against a target table and conditionally applies INSERT, UPDATE, or DELETE in a single atomic operation

Source ──► MERGE INTO Target

              │
              ├── Match found     → UPDATE or DELETE
              └── No match found  → INSERT


Why not just overwrite?

| Approach | Problem |
|---|---|
| Full overwrite | Expensive, loses history, not safe for concurrent reads |
| Append only | Creates duplicates, no updates possible |
| MERGE INTO | Targeted, atomic, safe — supports INSERT + UPDATE + DELETE |


### 2.  Incremental Load Strategies
###### Problem : no date column is present in dataset  how to extract data  based on date column?

###### Strategy Comparison

| Strategy | Use When | Reliable? |
|---|---|---|
| Watermark / ID Column | Append-only source, auto-increment ID | ✅ Fastest |
| CDC Table | Source emits Insert / Update / Delete events | ✅ Most accurate |
| Hash Comparison | No CDC, no ID — detect changes by hashing row | ✅ Good |
| File Metadata | File-based source, no DB access | ✅ For file pipelines |
| Snapshot Comparison | Full load available, compare old vs new | ⚠ Expensive |



###Case A — Watermark / ID Based (Append Only)
######Concept: Track the last processed max(id) in a watermark table. On every run, fetch only rows where id > last_processed_id.

```text
Watermark Table (table_name, last_processed_id)
                    ↓
Read last_processed_id
                    ↓
Source Query:
SELECT * 
FROM source_table
WHERE id > last_processed_id
                    ↓
Load Incremental Records
                    ↓
MERGE INTO target_table
(INSERT / UPDATE)
                    ↓
Get new MAX(id) from processed data
                    ↓
UPDATE Watermark Table
SET last_processed_id = new_max_id

•	Best for: Event logs, order tables, any append-only feed with a monotonically increasing key.
•	Not for: Tables where rows can be updated or deleted.




In [0]:
#Step 1: Preparing the source table 
column=['order_id', 'customer', 'amount']
data=[[1,"john wick",500],[2, "larry", 1000 ],[3,'shown',200]]
# data=[[3,"jack",500],[2, "Jorden", 1000 ],[3,'serena',200]]
source_df= spark.createDataFrame(data,column)
source_df.write.mode("overwrite").saveAsTable("source_order")
df=spark.sql("""
select * from source_order
""")
display(df)




In [0]:
# Step 2: create the target table with intital load
spark.sql(
    """
    create table if not exists target_order (
        order_id int,
        customer string,
        amount int
    )
    """
)

df_2=spark.sql("""
select * from target_order
""")
display(df_2)



In [0]:
# Step 3: Create watermark table
from pyspark.sql.functions import max 
spark.sql(
    """
    create table if not exists watermark_table as select "source_order" as table_name , max(order_id) as max_id from target_order
    """
)

df_3=spark.sql(
    """
    select max_id from watermark_table
    """
)
display(df_3)

rows = df_3.collect()

if len(rows) == 0 or rows[0]["max_id"] is None:
    last_processed_id = 0   # default start value
else:
    last_processed_id = rows[0]["max_id"]

print(last_processed_id)





In [0]:
# Step 4: merge the source with target table with new record 
print("merge in progress")
from delta.tables import DeltaTable
from pyspark.sql.functions import max
target_table = DeltaTable.forName(
    spark,"target_order"
)
source_df = spark.read.table("source_order")
incremental_df = source_df.filter(
    f"order_id > {last_processed_id}"
)
target_table.alias("t").merge(
    incremental_df.alias("s"),
    "t.order_id = s.order_id"
).whenNotMatchedInsert(
    # all = True
    values ={
        "order_id":"s.order_id",
        "customer" : "s.customer",
        "amount":"s.amount"
    }
).execute()

print("merge completed")



In [0]:
# Step 5: Updated the watermark
spark.sql("""
UPDATE watermark_table
SET max_id = COALESCE((
    SELECT MAX(order_id)
    FROM target_order
), 0)
WHERE table_name = 'source_order'
""")

df_3=spark.sql(
    """
    select * from watermark_table
    """
)
display(df_3)


In [0]:
spark.sql(
"""drop table watermark_table;
""")
spark.sql(
"""
drop table target_order;
""")
spark.sql(
"""
drop table source_order;""")

###Case B — CDC (Change Data Capture)
######Concept: Source system emits an operation column (Insert, Update, Delete) alongside each changed row. Merge handles all three in one pass.

| CDC Operation | MERGE Action | Notes |
|---|---|---|
| `operation = 'Insert'` | `whenNotMatchedInsert` | ⚠ Always add `condition="s.operation = 'Insert'"` |
| `operation = 'Update'` | `whenMatchedUpdate` | Updates existing matching rows |
| `operation = 'Delete'` | `whenMatchedDelete` | Deletes existing matching rows |

### Critical Note

Without:

```python
condition = "s.operation = 'Insert'"

•	Best for: Databases with CDC enabled (Debezium, SQL Server CDC, Oracle GoldenGate).




In [0]:
#Step 1: Preparing the source table 
column=['order_id', 'customer', 'amount','operation']
data=[[1,"john wick",500,'Insert'],[2, "larry", 1000,'Update' ],[3,'shown',200,'Delete']]
# data=[[3,"jack",500],[2, "Jorden", 1000 ],[3,'serena',200]]
source_df= spark.createDataFrame(data,column)
source_df.write.mode("overwrite").option("mergeSchema", True).saveAsTable("source_order_cdc")
df=spark.sql("""
select * from source_order_cdc
""")
display(df)

In [0]:
# Step 2: create the target table with intital load
spark.sql(
    """
    create table if not exists target_order (
        order_id int,
        customer string,
        amount int
    )
    """
)
spark.sql(
    """
    insert into workspace.default.target_order
    values(
        (3,'shown',200)
    )
    """
)

df_2=spark.sql("""
select * from workspace.default.target_order
""")
display(df_2)

In [0]:
from delta.tables import DeltaTable
cdc_df=spark.read.table("source_order_cdc")

target_table = DeltaTable.forName(spark, "target_order")
target_table.alias("t").merge(
    cdc_df.alias("s"),
    "t.order_id = s.order_id"
)\
.whenMatchedUpdate(
    
    condition="s.operation = 'Update'",
    set={
        'customer':'s.customer',
        'amount':'s.amount'
    }
    
)\
.whenMatchedDelete(
    condition="s.operation = 'Delete'"
    
)\
.whenNotMatchedInsert(
    values ={
        "order_id":"s.order_id",
        "customer" : "s.customer",
        "amount":"s.amount"
    }
  
).execute()
print("Merge completed successfully")

In [0]:
df_2=spark.sql("""
select * from workspace.default.target_order
""")
display(df_2)

In [0]:

spark.sql(
"""
drop table target_order;
""")
spark.sql(
"""
drop table source_order_cdc;""")

#### Hash Comparison

In [0]:
from pyspark.sql.functions import md5, concat_ws
def create_hash(l):
    print(*l)
    return md5(
        concat_ws(
            "||",
            *l
        )
    )

In [0]:
#Step 1: Preparing the source table 
column=['order_id', 'customer', 'amount']
data=[[1,"john wick",500],[2, "larry", 1000],[3,'shown',200]]
# data=[[3,"jack",500],[2, "Jorden", 1000 ],[3,'serena',200]]
source_df= spark.createDataFrame(data,column)
source_df=source_df.withColumn('hash',create_hash(source_df))
source_df.write.mode("overwrite").option("mergeSchema", True).saveAsTable("source_order_cdc")
df=spark.sql("""
select * from source_order_cdc
""")
display(df)

In [0]:
# Step 2: create the target table with intital load

column=['order_id', 'customer', 'amount']
data=[[2, "larry", 1000],[3,'shown',200]]
# data=[[3,"jack",500],[2, "Jorden", 1000 ],[3,'serena',200]]
target_df= spark.createDataFrame(data,column)
target_df=target_df.withColumn('hash',create_hash(target_df))
target_df.write.mode("overwrite").option("mergeSchema", True).saveAsTable("target_orders")


df_2=spark.sql("""
select * from workspace.default.target_orders
""")
display(df_2)

In [0]:
from delta.tables import DeltaTable
has_df = spark.read.table("source_order_cdc")
target = DeltaTable.forName(spark, "target_orders")
(
    target.alias("t").merge(
        has_df.alias("s"),
        "t.order_id = s.order_id",
    )
    .whenMatchedUpdate(
        condition="s.hash <> t.hash",
        set ={
             "customer": "s.customer",
            "amount": "s.amount",
            "hash": "s.hash"
        }
   )
    .whenNotMatchedInsert(
        values ={
            "order_id":"s.order_id",
            "customer" : "s.customer",
            "amount": "s.amount",
            "hash":"s.hash"
        }
)
    .whenNotMatchedBySourceDelete()
    .execute()
)
print("merge completed successfully")


In [0]:
%sql
drop table target_orders


In [0]:
%sql
drop table source_order_cdc

###Case C — File Metadata Based
######Concept: Track either the last processed date or a list of processed filenames. On each run, skip already-processed files.
Two approaches:
  1. Date-based  → file_metadata table stores last_processed_date
  2. File-based  → processed_files table stores file_path per file
  
•	Best for: Landing zone patterns where files arrive by date partition or with unique filenames.


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS file_metadata (
    source_name STRING,
    last_processed_date STRING
)
""")

In [0]:
spark.sql("""
INSERT INTO file_metadata
VALUES ('orders_data', '2026-05-14')
""")

In [0]:
metadata_df = spark.table("file_metadata")

last_processed_date = (
    metadata_df
    .filter("source_name = 'orders_data'")
    .select("last_processed_date")
    .collect()[0][0]
)

print(last_processed_date)

In [0]:
df = spark.read.format("csv") \
    .option("header", True) \
    .load("/Volumes/workspace/default/assignment/load_date=2026-05-15.csv")

In [0]:
df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("target_orders")

In [0]:
spark.sql("""
UPDATE file_metadata
SET last_processed_date = '2026-05-15'
WHERE source_name = 'orders_data'
""")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS processed_files (
    file_name STRING
)
""")

In [0]:
df = spark.read.format("csv") \
    .option("header", True) \
    .load("/Volumes/workspace/default/assignment/merge/*.csv")

In [0]:
from pyspark.sql.functions import input_file_name,col

df = df.withColumn(
    "file_name",
    # input_file_name()
    col("_metadata.file_path")
)

In [0]:
processed_df = spark.table("processed_files")

In [0]:
new_files_df = df.join(
    processed_df,
    on="file_name",
    how="left_anti"
)

In [0]:
# new_files_df.write \
#     .format("delta") \
#     .mode("append") \
#     .saveAsTable("target_order", mode="overwrite")
from delta.tables import DeltaTable

# Create target table if it doesn't exist
if not spark.catalog.tableExists("target_order"):
    new_files_df.drop("file_name").write \
        .format("delta") \
        .saveAsTable("target_order")
else:
    # Use merge for existing table
    target_table = DeltaTable.forName(spark, "target_order")
    
    target_table.alias("t").merge(
        new_files_df.drop("file_name").alias("s"),
        "t.order_id = s.order_id"  # Match on order_id
    ) \
    .whenMatchedUpdate(
        set={
            "customer": "s.customer",
            "amount": "s.amount"
        }
    ) \
    .whenNotMatchedInsert(
        values={
            "order_id": "s.order_id",
            "customer": "s.customer",
            "amount": "s.amount"
        }
    ) \
    .execute()

print("Merge completed successfully")

In [0]:
new_files_df.select("file_name") \
    .distinct() \
    .write \
    .mode("append") \
    .saveAsTable("processed_files")

In [0]:
%sql
drop table target_orders


In [0]:
%sql
drop table processed_files 


In [0]:
%sql
drop table file_metadata

###3.  Merge Without a Natural Key
###### problem: Source has no single unique ID column. How do you define what 'match' means?


| Strategy | Common? | Reliable? | Notes |
|---|---|---|---|
| Composite Key | ✅ Very common | ✅ Good | Combine 2-3 stable columns |
| Full Row Hash | ✅ Common | ⚠ Depends | Hash all business columns |
| Surrogate Key | ⚠ Shared only | ✅ Controlled | Must NOT be generated independently |
| Window + Row Number | ⚠ Limited | ❌ Risky | Row order must be deterministic |
| Fuzzy Matching | Rare | ⚠ Expensive | Levenshtein — avoid on large tables |

####Composite Key
######Concept: Concatenate 2-3 stable columns (e.g., name + city) to form a synthetic key.


<div style="background-color:#F8D7DA;
            padding:15px;
            border-radius:10px;
            border-left:6px solid #F8D7DA;
            font-size:15px;">

<pre>
composite_key = concat_ws('||', name, city)

⚠ Deduplicate source BEFORE merge.
If two rows share the same composite key,
keep the latest by updated_at timestamp
using Window + row_number().
</pre>

</div>




In [0]:
#######################################Compisite key ############################################


# ============================================
# STEP 1 — CREATE SOURCE DATAFRAME
# ============================================

source_data = [
    ("Ajit",  "Delhi",     500),
    ("Rahul", "Mumbai",   1000),
    ("Neha",  "Pune",      700),
    ("Ajit",  "Delhi",     800),   # updated amount
    ("Aman",  "Chennai",   900)
]

source_columns = [
    "name",
    "city",
    "amount"
]

source_df = spark.createDataFrame(
    source_data,
    source_columns
)

display(source_df)



In [0]:
# ============================================
# STEP 2 — SAVE SOURCE AS TABLE
# ============================================

source_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.source_orders")

In [0]:
# ============================================
# STEP 3 — CREATE TARGET TABLE
# ============================================

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.target_orders (
    name STRING,
    city STRING,
    amount LONG
)
""")

In [0]:
spark.table("workspace.default.target_orders").printSchema()
target_df.printSchema()

In [0]:
# ============================================
# STEP 4 — INSERT INITIAL DATA INTO TARGET
# ============================================

target_data = [
    ("Ajit",  "Delhi",   400),
    ("Rahul", "Mumbai",  1000),
    ("Karan", "Noida",    300)
]

target_columns = [
    "name",
    "city",
    "amount"
]

target_df = spark.createDataFrame(
    target_data,
    target_columns
)

target_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.default.target_orders")

In [0]:
# ============================================
# STEP 5 — READ SOURCE TABLE
# ============================================

source_df = spark.table(
    "workspace.default.source_orders"
)

display(source_df)

In [0]:
# ============================================
# STEP 6 — CREATE COMPOSITE KEY
# ============================================

from pyspark.sql.functions import concat_ws

source_df = source_df.withColumn(
    "composite_key",
    concat_ws(
        "||",
        "name",
        "city"
    )
)

display(source_df)

In [0]:
# ============================================
# STEP 7 — ADD COMPOSITE KEY TO TARGET
# ============================================

target_df = spark.table(
    "workspace.default.target_orders"
)

target_df = target_df.withColumn(
    "composite_key",
    concat_ws(
        "||",
        "name",
        "city"
    )
)

# overwrite target table with key

target_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.target_orders")

In [0]:
display(spark.table("workspace.default.target_orders"))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("composite_key") \
    .orderBy(col("amount").desc())

source_df = source_df.withColumn(
    "rn",
    row_number().over(window_spec)
).filter(
    col("rn") == 1
).drop("rn")

display(source_df)

In [0]:
# ============================================
# STEP 8 — PERFORM MERGE
# ============================================

from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.default.target_orders"
)

(
    target.alias("t")
    .merge(
        source_df.alias("s"),
        """
        t.composite_key = s.composite_key
        """
    )

    # UPDATE existing rows
    .whenMatchedUpdate(
        set={
            "amount": "s.amount"
        }
    )

    # INSERT new rows
    .whenNotMatchedInsert(
        values={
            "name": "s.name",
            "city": "s.city",
            "amount": "s.amount",
            "composite_key": "s.composite_key"
        }
    )

    .execute()
)

In [0]:
# ============================================
# STEP 9 — VIEW FINAL TARGET TABLE
# ============================================

final_df = spark.table(
    "workspace.default.target_orders"
)

display(final_df)

In [0]:
#Droping tables 

spark.sql("DROP TABLE IF EXISTS workspace.default.source_orders")

spark.sql("DROP TABLE IF EXISTS workspace.default.target_orders")



####Full Row Hash
######Concept: Hash all business columns into a single hash column. Use it as both a match key and a change-detection column.



<div style="background-color:#F8D7DA;
            padding:15px;
            border-radius:10px;
            border-left:6px solid #F8D7DA;
            font-size:15px;">

<pre>
hash = md5(concat_ws('||', col1, col2, col3))
Match on  : t.order_id = s.order_id
Update when: t.hash <> s.hash  (data actually changed)
</pre>

</div>




In [0]:
# Steps will be same only in place of composite key we will add hash key by creating hash 

####Surrogate Key

######Concept: Use a system-assigned unique ID that travels with the record from source to target.

<div style="background-color:#F8D7DA;
            padding:15px;
            border-radius:10px;
            border-left:6px solid #F8D7DA;
            font-size:15px;">

<pre>

⚠  Never generate surrogate keys independently per DataFrame using monotonically_increasing_id(). IDs generated in separate Spark jobs have no relationship to each other.

✅  Always derive the target's surrogate key by joining back to the source table on natural columns (name + city).
</pre>

</div>


In [0]:
####################################### Surrogate key ############################################

# ============================================
# STEP 1 — CREATE SOURCE DATAFRAME
# ============================================

from datetime import datetime

source_data = [
    ("Ajit",  "Delhi",   500,  "2026-05-01 10:00:00"),  # older
    ("Rahul", "Mumbai", 1000,  "2026-05-01 09:00:00"),
    ("Neha",  "Pune",    700,  "2026-05-01 11:00:00"),
    ("Ajit",  "Delhi",   800,  "2026-05-02 15:00:00"),  # newer ← keep this
    ("Aman",  "Chennai", 900,  "2026-05-01 08:00:00")
]

source_columns = ["name", "city", "amount", "updated_at"]

source_df = spark.createDataFrame(source_data, source_columns)

display(source_df)



In [0]:
# ============================================
# STEP 2 — CREATE SURROGATE KEY
# ============================================

from pyspark.sql.functions import monotonically_increasing_id

source_df = source_df.withColumn(
    "surrogate_id",
    monotonically_increasing_id()
)

display(source_df)

In [0]:
#  DEDUPLICATE SOURCE (KEEP LATEST BY TIMESTAMP)

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window \
    .partitionBy("name", "city") \
    .orderBy(col("updated_at").desc())   # latest timestamp wins

source_df = source_df.withColumn(
    "rn",
    row_number().over(window_spec)
).filter(
    col("rn") == 1
).drop("rn")

display(source_df)

In [0]:
# ============================================
# STEP 3 — SAVE SOURCE TABLE
# ============================================

source_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.source_orders")

In [0]:
# ============================================
# STEP 4 — CREATE TARGET DATAFRAME
# ============================================

target_data = [
    ("Ajit",  "Delhi",   400),
    ("Rahul", "Mumbai", 1000),
    ("Karan", "Noida",   300)
]

target_columns = [
    "name",
    "city",
    "amount"
]

target_df = spark.createDataFrame(
    target_data,
    target_columns
)

display(target_df)

In [0]:
# ============================================
# STEP 5 — ADD SURROGATE KEY TO TARGET
# ============================================

# Surrogate keys must come from the source system, not be re-generated.
# Join target with the already-saved source table to pick up its IDs.

source_ref = spark.table("workspace.default.source_orders") \
    .select("name", "city", "surrogate_id")

target_df = target_df.join(
    source_ref,
    on=["name", "city"],
    how="left"          # Karan/Noida has no source match → surrogate_id = NULL (no update)
)

display(target_df)

In [0]:
# ============================================
# STEP 6 — SAVE TARGET TABLE
# ============================================

target_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.target_orders")

In [0]:
# ============================================
# STEP 7 — READ SOURCE TABLE
# ============================================

source_df = spark.table(
    "workspace.default.source_orders"
)

display(source_df)

In [0]:
# ============================================
# STEP 8 — PERFORM MERGE USING SURROGATE KEY
# ============================================

from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.default.target_orders"
)

(
    target.alias("t")
    .merge(
        source_df.alias("s"),
        """
        t.surrogate_id = s.surrogate_id
        """
    )

    # -----------------------------------
    # UPDATE EXISTING RECORDS
    # -----------------------------------

    .whenMatchedUpdate(
        set={
            "name": "s.name",
            "city": "s.city",
            "amount": "s.amount"
        }
    )

    # -----------------------------------
    # INSERT NEW RECORDS
    # -----------------------------------

    .whenNotMatchedInsert(
        values={
            "name": "s.name",
            "city": "s.city",
            "amount": "s.amount",
            "surrogate_id": "s.surrogate_id"
        }
    )

    .execute()
)

In [0]:
# ============================================
# STEP 9 — VIEW FINAL TARGET TABLE
# ============================================

final_df = spark.table(
    "workspace.default.target_orders"
)

display(final_df)

In [0]:
# ============================================
# Dropping the tables 
# ============================================

spark.sql("DROP TABLE IF EXISTS workspace.default.source_orders")

spark.sql("DROP TABLE IF EXISTS workspace.default.target_orders")

In [0]:
####################################### Window + Row Number ############################################
# ============================================
# STEP 1 — CREATE SOURCE DATAFRAME WITH TIMESTAMP
# ============================================

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from datetime import datetime

source_data = [
    ("Ajit",  "Delhi",     500,  datetime(2024, 1, 1, 10, 0, 0)),   # older record
    ("Rahul", "Mumbai",   1000,  datetime(2024, 1, 1, 9,  0, 0)),
    ("Neha",  "Pune",      700,  datetime(2024, 1, 1, 11, 0, 0)),
    ("Ajit",  "Delhi",     800,  datetime(2024, 1, 1, 12, 0, 0)),   # latest record ✅
    ("Aman",  "Chennai",   900,  datetime(2024, 1, 1, 8,  0, 0))
]

source_schema = StructType([
    StructField("name",       StringType(),    True),
    StructField("city",       StringType(),    True),
    StructField("amount",     IntegerType(),   True),
    StructField("updated_at", TimestampType(), True)   # ✅ timestamp column
])

source_df = spark.createDataFrame(source_data, source_schema)
display(source_df)

In [0]:
# ============================================
# STEP 2 — DEDUPLICATE SOURCE USING TIMESTAMP
# Keep the LATEST record per (name, city)
# ============================================

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

dedup_window = Window \
    .partitionBy("name", "city") \
    .orderBy(desc("updated_at"))        # ✅ Latest timestamp wins

source_deduped = (
    source_df
    .withColumn("rn", row_number().over(dedup_window))
    .filter("rn = 1")
    .drop("rn")                         # drop helper column
)

display(source_deduped)

In [0]:
# ============================================
# STEP 3 — SAVE SOURCE TABLE
# ============================================

source_deduped.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.source_orders")

In [0]:
# ============================================
# STEP 4 — CREATE TARGET DATAFRAME
# ============================================

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from datetime import datetime

target_data = [
    ("Ajit",  "Delhi",   400,  datetime(2024, 1, 1, 8, 0, 0)),
    ("Rahul", "Mumbai", 1000,  datetime(2024, 1, 1, 8, 0, 0)),
    ("Karan", "Noida",   300,  datetime(2024, 1, 1, 8, 0, 0))
]

target_schema = StructType([
    StructField("name",       StringType(),    True),
    StructField("city",       StringType(),    True),
    StructField("amount",     IntegerType(),   True),
    StructField("updated_at", TimestampType(), True)   # ✅ added
])

target_df = spark.createDataFrame(target_data, target_schema)

target_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.target_orders")

display(target_df)

In [0]:
# ============================================
# STEP 5 — READ SOURCE TABLE
# ============================================

source_df = spark.table("workspace.default.source_orders")
display(source_df)

In [0]:
# ============================================
# STEP 6 — PERFORM MERGE (WITHOUT DELETE)
# ============================================

from delta.tables import DeltaTable

source_df = spark.table("workspace.default.source_orders")

target = DeltaTable.forName(spark, "workspace.default.target_orders")

(
    target.alias("t")
    .merge(
        source_df.alias("s"),
        "t.name = s.name AND t.city = s.city"
    )

    # UPDATE existing records
    .whenMatchedUpdate(set={
        "amount":     "s.amount",
        "updated_at": "s.updated_at"
    })

    # INSERT new records
    .whenNotMatchedInsert(values={
        "name":       "s.name",
        "city":       "s.city",
        "amount":     "s.amount",
        "updated_at": "s.updated_at"
    })

    .execute()
)

In [0]:
# ============================================
# STEP 7 — VIEW FINAL TARGET TABLE
# ============================================

final_df = spark.table("workspace.default.target_orders")
display(final_df)

In [0]:
# ============================================
# Dropping tables 
# ============================================

spark.sql("DROP TABLE IF EXISTS workspace.default.source_orders")

spark.sql("DROP TABLE IF EXISTS workspace.default.target_orders")

####Fuzzy Matching
######Concept: Match rows where keys are similar but not identical using Levenshtein distance.
```levenshtein(t.name, s.name) <= 5 AND t.city = s.city```

<div style="background-color:#F8D7DA;
            padding:15px;
            border-radius:10px;
            border-left:6px solid #F8D7DA;
            font-size:15px;">

<pre>
⚠  Never use levenshtein() directly in the Delta merge condition on large tables — it generates a cartesian product. Pre-compute matches into a temp view first, then merge on a clean equi-join key.
</pre>

</div>

In [0]:
########################################### Fuzzy matching #############################################
# ============================================
# STEP 1 — CREATE SOURCE DATAFRAME
# ============================================

source_data = [
    ("Ajit Kumar",     "Delhi",    500),
    ("Rahul Sharma",   "Mumbai",  1000),
    ("Neha Verma",     "Pune",     700),
    ("Aman Singh",     "Chennai",  900),
    ("Ravi Patel",     "Noida",    300)
]

source_columns = [
    "name",
    "city",
    "amount"
]

source_df = spark.createDataFrame(
    source_data,
    source_columns
)

display(source_df)


In [0]:
# ============================================
# STEP 2 — SAVE SOURCE TABLE
# ============================================

source_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.source_customers")

In [0]:
# ============================================
# STEP 3 — CREATE TARGET DATAFRAME
# ============================================

target_data = [
    ("Ajit K.",        "Delhi",    400),
    ("Rahul Sharma",   "Mumbai",  1000),
    ("Neha V",         "Pune",     600)
]

target_columns = [
    "name",
    "city",
    "amount"
]

target_df = spark.createDataFrame(
    target_data,
    target_columns
)

display(target_df)

In [0]:
# ============================================
# STEP 4 — SAVE TARGET TABLE
# ============================================

target_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.target_customers")

In [0]:
# ============================================
# STEP 5 — READ SOURCE & TARGET TABLES
# ============================================

source_df = spark.table(
    "workspace.default.source_customers"
)

target_df = spark.table(
    "workspace.default.target_customers"
)

display(source_df)

display(target_df)

In [0]:
# ============================================
# STEP 6 — FUZZY MATCHING LOGIC
# ============================================

from pyspark.sql.functions import levenshtein

fuzzy_match_df = (
    source_df.alias("s")
    .join(
        target_df.alias("t"),
        (
            (levenshtein("s.name", "t.name") <= 5)
            &
            (source_df.city == target_df.city)
        ),
        "inner"
    )
)

display(fuzzy_match_df)

In [0]:
# ============================================
# STEP 7 — CREATE TEMP VIEW
# ============================================

fuzzy_match_df.createOrReplaceTempView(
    "matched_records"
)

In [0]:
# ============================================
# STEP 8 — PERFORM MERGE
# ============================================

from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.default.target_customers"
)

(
    target.alias("t")
    .merge(
        source_df.alias("s"),
        """
        levenshtein(t.name, s.name) <= 5
        AND t.city = s.city
        """
    )

    # -----------------------------------
    # UPDATE MATCHED RECORDS
    # -----------------------------------

    .whenMatchedUpdate(
        set={
            "amount": "s.amount"
        }
    )

    # -----------------------------------
    # INSERT NEW RECORDS
    # -----------------------------------

    .whenNotMatchedInsert(
        values={
            "name": "s.name",
            "city": "s.city",
            "amount": "s.amount"
        }
    )

    .execute()
)

In [0]:
# ============================================
# STEP 9 — VIEW FINAL TARGET TABLE
# ============================================

final_df = spark.table(
    "workspace.default.target_customers"
)

display(final_df)

In [0]:
# ============================================
#Dropping Table
# ============================================

spark.sql("DROP TABLE IF EXISTS workspace.default.source_customers")

spark.sql("DROP TABLE IF EXISTS workspace.default.target_customers")

####4.  Schema Evolution Handling
######Problem: Source sends a new column that doesn't exist in the target. What do you do?


%md

| Strategy | What Happens | When to Use |
|---|---|---|
| Ignore new column | Drop extra columns before merge | Safe default — strict governance |
| Auto-add column | autoMerge.enabled = true + UpdateAll/InsertAll | Flexible pipelines |
| Fail pipeline | Detect new columns, raise Exception | Strict contract enforcement |
| Add after approval | Validate against approved list, then evolve | Enterprise / regulated systems |
| Store separately | Serialize unknown columns into extra_payload (JSON) | Semi-structured sources |
| Version schema | Define schema contracts per version (v1, v2...) | Backward compatibility |

In [0]:
########################################### Ignore new column #############################################

# ============================================
# STEP 1 — CREATE SOURCE DATA (WITH EXTRA COLUMN)
# ============================================

source_data = [
    (1, "Ajit", 500, "Delhi"),
    (2, "Rahul", 1000, "Mumbai"),
    (3, "Neha", 700, "Pune")
]

source_columns = [
    "order_id",
    "customer",
    "amount",
    "city"   # NEW COLUMN (should be ignored)
]

source_df = spark.createDataFrame(source_data, source_columns)

display(source_df)


In [0]:
# ============================================
# STEP 2 — CREATE TARGET TABLE (WITHOUT CITY)
# ============================================

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.target_orders (
    order_id LONG,
    customer STRING,
    amount LONG
)
""")

In [0]:
source_df.printSchema()
spark.table("workspace.default.target_orders").printSchema()

In [0]:
# ============================================
# STEP 3 — INSERT INITIAL DATA INTO TARGET
# ============================================

target_data = [
    (1, "Ajit", 400),
    (2, "Rahul", 900)
]

target_columns = ["order_id", "customer", "amount"]

target_df = spark.createDataFrame(target_data, target_columns)

target_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.default.target_orders")

In [0]:
# ============================================
# STEP 4 — KEEP ONLY TARGET COLUMNS (IGNORE EXTRA)
# ============================================

expected_columns = ["order_id", "customer", "amount"]

filtered_source_df = source_df.select(*expected_columns)

display(filtered_source_df)

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.default.target_orders"
)

(
    target.alias("t")
    .merge(
        filtered_source_df.alias("s"),
        "t.order_id = s.order_id"
    )
    .whenMatchedUpdate(set={
        "customer": "s.customer",
        "amount": "s.amount"
    })
    .whenNotMatchedInsert(values={
        "order_id": "s.order_id",
        "customer": "s.customer",
        "amount": "s.amount"
    })
    .execute()
)

In [0]:
display(
    spark.table("workspace.default.target_orders")
)

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.default.source_orders")
spark.sql("DROP TABLE IF EXISTS workspace.default.target_orders")

In [0]:
%skip
########################################### Auto Add new column #############################################
spark.conf.set(
    "spark.databricks.delta.schema.autoMerge.enabled",
    "true"
)

In [0]:
# ============================================
# STEP 1 — SOURCE DATA (WITH NEW COLUMN)
# ============================================

source_data = [
    (1, "Ajit", 500, "Delhi"),
    (2, "Rahul", 1000, "Mumbai"),
    (3, "Neha", 700, "Pune")
]

source_columns = [
    "order_id",
    "customer",
    "amount",
    "city"   # NEW COLUMN
]

source_df = spark.createDataFrame(source_data, source_columns)

display(source_df)

In [0]:
# ============================================
# STEP 2 — TARGET TABLE (OLD SCHEMA ONLY)
# ============================================

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.target_orders (
    order_id LONG,
    customer STRING,
    amount LONG
)
""")

In [0]:
source_df.printSchema()
spark.table("workspace.default.target_orders").printSchema()

In [0]:
# ============================================
# STEP 3 — INITIAL LOAD INTO TARGET
# ============================================

target_data = [
    (1, "Ajit", 400),
    (2, "Rahul", 900)
]

target_columns = ["order_id", "customer", "amount"]

target_df = spark.createDataFrame(target_data, target_columns)

target_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.default.target_orders")

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.default.target_orders"
)

(
    target.alias("t")
    .merge(
        source_df.alias("s"),
        "t.order_id = s.order_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
display(
    spark.table("workspace.default.target_orders")
)

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.default.source_orders")
spark.sql("DROP TABLE IF EXISTS workspace.default.target_orders")

In [0]:
########################################### Fail pipeline #############################################
# ============================================
# STEP 1 — SOURCE DATA (WITH NEW COLUMN)
# ============================================

source_data = [
    (1, "Ajit", 500, "Delhi"),
    (2, "Rahul", 1000, "Mumbai"),
    (3, "Neha", 700, "Pune")
]

source_columns = [
    "order_id",
    "customer",
    "amount",
    "city"   # NEW COLUMN (not allowed)
]

source_df = spark.createDataFrame(source_data, source_columns)

display(source_df)

In [0]:
# ============================================
# STEP 2 — TARGET SCHEMA (EXPECTED CONTRACT)
# ============================================

expected_columns = ["order_id", "customer", "amount"]

target_data = [
    (1, "Ajit", 400),
    (2, "Rahul", 900)
]

target_df = spark.createDataFrame(target_data, expected_columns)

target_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.target_orders")

In [0]:
# ============================================
# STEP 3 — DETECT NEW / UNEXPECTED COLUMNS
# ============================================

source_columns_set = set(source_df.columns)
expected_columns_set = set(expected_columns)

new_columns = list(source_columns_set - expected_columns_set)

print("New Columns Detected:", new_columns)

In [0]:
# ============================================
# STEP 4 — STRICT SCHEMA CHECK (FAIL FAST)
# ============================================

if len(new_columns) > 0:
    raise Exception(
        f"""
        ❌ PIPELINE FAILED DUE TO SCHEMA CHANGE

        New columns detected: {new_columns}

        Expected schema: {expected_columns}

        Action required:
        - Update schema contract OR
        - Reject source changes OR
        - Get approval before ingestion
        """
    )

In [0]:
# ============================================
# STEP 5 — FILTER ONLY ALLOWED COLUMNS
# ============================================

filtered_source_df = source_df.select(*expected_columns)

In [0]:
# ============================================
# STEP 6 — READ TARGET TABLE
# ============================================

target_df = spark.table("workspace.default.target_orders")

In [0]:
# ============================================
# STEP 7 — PERFORM MERGE (SAFE EXECUTION)
# ============================================

from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.default.target_orders"
)

(
    target.alias("t")
    .merge(
        filtered_source_df.alias("s"),
        "t.order_id = s.order_id"
    )
    .whenMatchedUpdate(set={
        "customer": "s.customer",
        "amount": "s.amount"
    })
    .whenNotMatchedInsert(values={
        "order_id": "s.order_id",
        "customer": "s.customer",
        "amount": "s.amount"
    })
    .execute()
)

In [0]:
# ============================================
# STEP 8 — FINAL OUTPUT
# ============================================

display(
    spark.table("workspace.default.target_orders"))

In [0]:

spark.sql("DROP TABLE IF EXISTS workspace.default.source_orders")
spark.sql("DROP TABLE IF EXISTS workspace.default.target_orders")

In [0]:
########################################### Add after approval #############################################

In [0]:
# ============================================
# STEP 1 — CREATE SOURCE DATA
# ============================================

# Source has NEW COLUMN -> city

source_data = [
    (1, "Ajit", 500, "Delhi"),
    (2, "Rahul", 1000, "Mumbai"),
    (3, "Neha", 700, "Pune")
]

source_columns = [
    "order_id",
    "customer",
    "amount",
    "city"     # NEW COLUMN
]

source_df = spark.createDataFrame(
    source_data,
    source_columns
)

display(source_df)

In [0]:
# ============================================
# STEP 2 — CREATE TARGET TABLE
# ============================================

# OLD SCHEMA (WITHOUT city)

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.target_orders (
    order_id LONG,
    customer STRING,
    amount LONG
)
""")

In [0]:
# ============================================
# STEP 3 — INSERT INITIAL TARGET DATA
# ============================================

target_data = [
    (1, "Ajit", 400),
    (2, "Rahul", 900)
]

target_columns = [
    "order_id",
    "customer",
    "amount"
]

target_df = spark.createDataFrame(
    target_data,
    target_columns
)

target_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.default.target_orders")

In [0]:
# ============================================
# STEP 4 — DEFINE EXPECTED SCHEMA
# ============================================

expected_columns = [
    "order_id",
    "customer",
    "amount"
]

In [0]:
# ============================================
# STEP 5 — DETECT NEW COLUMNS
# ============================================

source_columns = source_df.columns

new_columns = list(
    set(source_columns) - set(expected_columns)
)

print("New Columns Detected:", new_columns)

In [0]:
# ============================================
# STEP 6 — APPROVAL LIST
# ============================================

# Only approved columns can be added

approved_new_columns = [
    "city"
]

In [0]:
# ============================================
# STEP 7 — VALIDATE NEW COLUMNS
# ============================================

unapproved_columns = list(
    set(new_columns) - set(approved_new_columns)
)

print("Unapproved Columns:", unapproved_columns)

In [0]:
# ============================================
# STEP 8 — FAIL IF UNAPPROVED COLUMN EXISTS
# ============================================

if len(unapproved_columns) > 0:

    raise Exception(
        f"""
        ❌ PIPELINE FAILED

        Unapproved columns detected:
        {unapproved_columns}

        Approved columns:
        {approved_new_columns}
        """
    )

else:

    print("✅ All new columns are approved")

In [0]:
%skip
# ============================================
# STEP 9 — ENABLE AUTO MERGE
# ============================================

spark.conf.set(
    "spark.databricks.delta.schema.autoMerge.enabled",
    "true"
)

In [0]:
# ============================================
# STEP 10 — BUILD FINAL ALLOWED COLUMNS
# ============================================

final_allowed_columns = (
    expected_columns +
    approved_new_columns
)

print(final_allowed_columns)

In [0]:
# ============================================
# STEP 11 — FILTER SOURCE DATA
# ============================================

filtered_source_df = source_df.select(
    *final_allowed_columns
)

display(filtered_source_df)

In [0]:
# ============================================
# STEP 12 — PERFORM MERGE
# ============================================

from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.default.target_orders"
)

(
    target.alias("t")

    .merge(
        filtered_source_df.alias("s"),
        "t.order_id = s.order_id"
    )
    .withSchemaEvolution()
    .whenMatchedUpdateAll()

    .whenNotMatchedInsertAll()

    .execute()
)

In [0]:
# ============================================
# STEP 13 — VIEW FINAL TABLE
# ============================================

final_df = spark.table(
    "workspace.default.target_orders"
)

display(final_df)

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.default.source_orders")
spark.sql("DROP TABLE IF EXISTS workspace.default.target_orders")

In [0]:
########################################### Store separately #############################################
# ============================================
# STEP 1 — CREATE SOURCE DATA
# ============================================

# Source contains NEW / UNKNOWN columns

source_data = [
    (1, "Ajit", 500, "Delhi", "A"),
    (2, "Rahul", 1000, "Mumbai", "B"),
    (3, "Neha", 700, "Pune", "C")
]

source_columns = [
    "order_id",
    "customer",
    "amount",
    "city",          # unknown column
    "temp_flag"      # unknown column
]

source_df = spark.createDataFrame(
    source_data,
    source_columns
)

display(source_df)

In [0]:
# ============================================
# STEP 2 — CREATE TARGET TABLE
# ============================================

# Structured warehouse schema

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.target_orders (
    order_id LONG,
    customer STRING,
    amount LONG,
    extra_payload STRING
)
""")

In [0]:
# ============================================
# STEP 3 — INSERT INITIAL TARGET DATA
# ============================================
from pyspark.sql.types import *
target_data = [
    (1, "Ajit", 400, None),
    (2, "Rahul", 900, None)
]
schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("customer", StringType(), True),
    StructField("amount", LongType(), True),
    StructField("extra_payload", StringType(), True)
])


target_columns = [
    "order_id",
    "customer",
    "amount",
    "extra_payload"
]

target_df = spark.createDataFrame(
    target_data,
    schema
)

target_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.default.target_orders")

In [0]:
# ============================================
# STEP 4 — DEFINE EXPECTED COLUMNS
# ============================================

expected_columns = [
    "order_id",
    "customer",
    "amount"
]

In [0]:
# ============================================
# STEP 5 — DETECT EXTRA / UNKNOWN COLUMNS
# ============================================

source_columns = source_df.columns

extra_columns = list(
    set(source_columns) - set(expected_columns)
)

print("Extra Columns:", extra_columns)

In [0]:
# ============================================
# STEP 6 — CREATE STRUCTURED DATAFRAME
# ============================================

# Keep only governed columns

structured_df = source_df.select(
    *expected_columns
)

display(structured_df)

In [0]:
# ============================================
# STEP 7 — CREATE EXTRA PAYLOAD COLUMN
# ============================================

from pyspark.sql.functions import to_json, struct

payload_df = source_df.withColumn(
    "extra_payload",
    to_json(
        struct(*extra_columns)
    )
)

display(payload_df)

In [0]:
# ============================================
# STEP 8 — ATTACH EXTRA PAYLOAD
# ============================================

final_df = (
    structured_df.alias("s")

    .join(
        payload_df.select(
            "order_id",
            "extra_payload"
        ).alias("p"),

        on="order_id",
        how="left"
    )
)

display(final_df)

In [0]:
# ============================================
# STEP 9 — PERFORM MERGE
# ============================================

from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.default.target_orders"
)

(
    target.alias("t")

    .merge(
        final_df.alias("s"),
        "t.order_id = s.order_id"
    )

    .whenMatchedUpdate(
        set={
            "customer": "s.customer",
            "amount": "s.amount",
            "extra_payload": "s.extra_payload"
        }
    )

    .whenNotMatchedInsert(
        values={
            "order_id": "s.order_id",
            "customer": "s.customer",
            "amount": "s.amount",
            "extra_payload": "s.extra_payload"
        }
    )

    .execute()
)

In [0]:
# ============================================
# STEP 10 — VIEW FINAL TABLE
# ============================================

final_target_df = spark.table(
    "workspace.default.target_orders"
)

display(final_target_df)

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.default.source_orders")
spark.sql("DROP TABLE IF EXISTS workspace.default.target_orders")

In [0]:
########################################### Version schema #############################################
# ============================================
# STEP 1 — CREATE SOURCE DATA (SCHEMA VERSION 2)
# ============================================

# New schema version contains:
# city column

source_data = [
    (1, "Ajit", 500, "Delhi"),
    (2, "Rahul", 1000, "Mumbai"),
    (3, "Neha", 700, "Pune")
]

source_columns = [
    "order_id",
    "customer",
    "amount",
    "city"
]

source_df = spark.createDataFrame(
    source_data,
    source_columns
)

display(source_df)

In [0]:
# ============================================
# STEP 2 — DEFINE SCHEMA VERSION
# ============================================

# Source tells pipeline which schema version it follows

schema_version = "v2"

print("Incoming Schema Version:", schema_version)

In [0]:
# ============================================
# STEP 3 — CREATE TARGET TABLE
# ============================================

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.target_orders (
    order_id LONG,
    customer STRING,
    amount LONG,
    city STRING,
    schema_version STRING
)
""")

In [0]:
# ============================================
# STEP 4 — INSERT INITIAL OLD DATA (VERSION 1)
# ============================================

target_data = [
    (1, "Ajit", 400, None, "v1"),
    (2, "Rahul", 900, None, "v1")
]
from pyspark.sql.types import *

schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("customer", StringType(), True),
    StructField("amount", LongType(), True),
    StructField("city", StringType(), True),
    StructField("schema_version", StringType(), True)
])

target_data = [
    (1, "Ajit", 400, None, "v1"),
    (2, "Rahul", 900, None, "v1")
]

target_df = spark.createDataFrame(target_data, schema)
target_columns = [
    "order_id",
    "customer",
    "amount",
    "city",
    "schema_version"
]

target_df = spark.createDataFrame(
    target_data,
    schema
)

target_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.default.target_orders")

In [0]:
# ============================================
# STEP 5 — DEFINE SCHEMA CONTRACTS
# ============================================

schema_contracts = {

    "v1": [
        "order_id",
        "customer",
        "amount"
    ],

    "v2": [
        "order_id",
        "customer",
        "amount",
        "city"
    ]
}

In [0]:
# ============================================
# STEP 6 — GET EXPECTED COLUMNS
# ============================================

expected_columns = schema_contracts[
    schema_version
]

print(expected_columns)

In [0]:
# ============================================
# STEP 7 — VALIDATE SOURCE SCHEMA
# ============================================

source_columns = source_df.columns

unexpected_columns = list(
    set(source_columns) - set(expected_columns)
)

missing_columns = list(
    set(expected_columns) - set(source_columns)
)

print("Unexpected Columns:", unexpected_columns)

print("Missing Columns:", missing_columns)

In [0]:
# ============================================
# STEP 8 — FAIL IF SCHEMA INVALID
# ============================================

if len(unexpected_columns) > 0:

    raise Exception(
        f"""
        ❌ INVALID SCHEMA VERSION

        Unexpected columns:
        {unexpected_columns}
        """
    )

if len(missing_columns) > 0:

    raise Exception(
        f"""
        ❌ MISSING REQUIRED COLUMNS

        Missing columns:
        {missing_columns}
        """
    )

print("✅ Schema validation passed")

In [0]:
# ============================================
# STEP 9 — ADD SCHEMA VERSION COLUMN
# ============================================

from pyspark.sql.functions import lit

source_df = source_df.withColumn(
    "schema_version",
    lit(schema_version)
)

display(source_df)

In [0]:
%skip
# ============================================
# STEP 10 — ENABLE AUTO MERGE
# ============================================

spark.conf.set(
    "spark.databricks.delta.schema.autoMerge.enabled",
    "true"
)

In [0]:
# ============================================
# STEP 11 — PERFORM MERGE
# ============================================

from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.default.target_orders"
)

(
    target.alias("t")

    .merge(
        source_df.alias("s"),
        "t.order_id = s.order_id"
    )
    .withSchemaEvolution()
    .whenMatchedUpdateAll()

    .whenNotMatchedInsertAll()

    .execute()
)

In [0]:
# ============================================
# STEP 12 — VIEW FINAL TABLE
# ============================================

final_df = spark.table(
    "workspace.default.target_orders"
)

display(final_df)

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.default.source_orders")
spark.sql("DROP TABLE IF EXISTS workspace.default.target_orders")

#####5.  File Format Handling
######Golden Rule: Never merge raw source files directly into the target.


| Never directly merge raw source data into target. |
|--------|
| Raw Files |
|     ↓ |
| Bronze Layer (raw ingestion) |
|     ↓  |
| Validation / Cleansing |
|     ↓ |
| Quarantine Bad Records |
|     ↓ |
| Silver Layer (clean data) |
|     ↓  |
| MERGE into Target |

######CSV — Malformed Records

•	Use PERMISSIVE mode (default) — bad rows become NULL instead of failing pipeline

•	Use badRecordsPath to capture raw bad lines for investigation

•	Filter NULLs → quarantine table;  clean rows → merge


In [0]:
####### Reading Maleformed CSV ##################

In [0]:
# ============================================
# STEP 1 — CREATE SAMPLE CORRUPTED CSV
# ============================================

# Assume CSV contains:
#
# order_id,customer,amount
# 1,Ajit,500
# 2,Rahul,abc
# 3,Neha,700
# 4,,1000

csv_path = "/Volumes/workspace/default/assignment/csv/csv_bad.csv"

In [0]:
# ============================================
# STEP 2 — DEFINE SCHEMA
# ============================================

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)

schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer", StringType(), True),
    StructField("amount", IntegerType(), True)
])

In [0]:
# ============================================
# STEP 3 — READ RAW FILE
# ============================================

# PERMISSIVE mode:
# Bad rows become NULL instead of failing

raw_df = (
    spark.read
    .format("csv")
    .option("header", True)
    # .option("mode", "PERMISSIVE")
    # .option(
    #     "badRecordsPath",
    #     "/Volumes/workspace/default/assignment/bad_records/"
    # )
    .schema(schema)
    .load(csv_path)
)

display(raw_df)

In [0]:
# ============================================
# STEP 4 — CREATE TARGET TABLE
# ============================================

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.target_orders (
    order_id LONG,
    customer STRING,
    amount LONG
)
""")

In [0]:
# ============================================
# STEP 5 — CREATE QUARANTINE TABLE
# ============================================

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.quarantine_orders (
    order_id INT,
    customer STRING,
    amount INT,
    error_reason STRING
)
""")

In [0]:
# ============================================
# STEP 6 — SEPARATE BAD RECORDS
# ============================================

from pyspark.sql.functions import lit

bad_df = (
    raw_df
    .filter(
        """
        order_id IS NULL
        OR customer IS NULL
        OR amount IS NULL
        """
    )
    .withColumn(
        "error_reason",
        lit("Invalid or corrupted record")
    )
)

display(bad_df)

In [0]:
bad_df.printSchema()
spark.table("workspace.default.quarantine_orders").printSchema()

In [0]:
# ============================================
# STEP 7 — STORE BAD RECORDS
# ============================================

bad_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        "workspace.default.quarantine_orders"
    )

In [0]:
# ============================================
# STEP 8 — FILTER CLEAN RECORDS
# ============================================

clean_df = raw_df.filter(
    """
    order_id IS NOT NULL
    AND customer IS NOT NULL
    AND amount IS NOT NULL
    """
)

display(clean_df)

In [0]:
# ============================================
# STEP 9 — INSERT INITIAL TARGET DATA
# ============================================

initial_data = [
    (1, "Ajit", 400)
]

initial_columns = [
    "order_id",
    "customer",
    "amount"
]

initial_df = spark.createDataFrame(
    initial_data,
    initial_columns
)

initial_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        "workspace.default.target_orders"
    )

In [0]:
# ============================================
# STEP 10 — PERFORM MERGE
# ============================================

from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.default.target_orders"
)

(
    target.alias("t")

    .merge(
        clean_df.alias("s"),
        "t.order_id = s.order_id"
    )

    .whenMatchedUpdate(
        set={
            "customer": "s.customer",
            "amount": "s.amount"
        }
    )

    .whenNotMatchedInsert(
        values={
            "order_id": "s.order_id",
            "customer": "s.customer",
            "amount": "s.amount"
        }
    )

    .execute()
)

In [0]:
# ============================================
# STEP 11 — VIEW FINAL TARGET TABLE
# ============================================

display(
    spark.table(
        "workspace.default.target_orders"
    )
)

In [0]:
# ============================================
# STEP 12 — VIEW QUARANTINE TABLE
# ============================================

display(
    spark.table(
        "workspace.default.quarantine_orders"
    )
)

In [0]:
# ============================================
# DROP TABLES (DEMO CLEANUP)
# ============================================

spark.sql("""
DROP TABLE IF EXISTS workspace.default.target_orders
""")

spark.sql("""
DROP TABLE IF EXISTS workspace.default.quarantine_orders
""")

######CSV — Malformed Records

•	Use PERMISSIVE mode (default) — bad rows become NULL instead of failing pipeline

•	Use badRecordsPath to capture raw bad lines for investigation

•	Filter NULLs → quarantine table;  clean rows → merge


In [0]:
################  CSV More than one Headers (Duplicate Headers ) ################



In [0]:
# ============================================
# STEP 1 — READ CSV AS STRING
# ============================================

# Read everything as STRING first
# to safely detect duplicate headers

raw_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", False)
    .load("/Volumes/workspace/default/assignment/csv/csv_bad - Copy.csv")
)

display(raw_df)

In [0]:
# ============================================
# STEP 2 — REMOVE DUPLICATE HEADER ROWS
# ============================================

clean_header_df = raw_df.filter(
    ~(
        (raw_df.order_id == "order_id")
        &
        (raw_df.customer == "customer")
        &
        (raw_df.amount == "amount")
    )
)

display(clean_header_df)

In [0]:
# ============================================
# STEP 3 — CAST TO CORRECT TYPES
# ============================================

from pyspark.sql.functions import col

typed_df = (
    clean_header_df

    .withColumn(
        "order_id",
        col("order_id").cast("int")
    )

    .withColumn(
        "amount",
        col("amount").cast("int")
    )
)

display(typed_df)

In [0]:
# ============================================
# STEP 4 — CREATE TARGET TABLE
# ============================================

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.target_orders (
    order_id INT,
    customer STRING,
    amount INT
)
""")

In [0]:
# ============================================
# STEP 5 — INSERT INITIAL DATA
# ============================================

initial_data = [
    (1, "Ajit", 400)
]

initial_columns = [
    "order_id",
    "customer",
    "amount"
]

initial_df = spark.createDataFrame(
    initial_data,
    initial_columns
)

initial_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        "workspace.default.target_orders"
    )

In [0]:
# ============================================
# STEP 6 — PERFORM MERGE
# ============================================

from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.default.target_orders"
)

(
    target.alias("t")

    .merge(
        typed_df.alias("s"),
        "t.order_id = s.order_id"
    )

    .whenMatchedUpdate(
        set={
            "customer": "s.customer",
            "amount": "s.amount"
        }
    )

    .whenNotMatchedInsert(
        values={
            "order_id": "s.order_id",
            "customer": "s.customer",
            "amount": "s.amount"
        }
    )

    .execute()
)

In [0]:
# ============================================
# STEP 7 — VIEW FINAL OUTPUT
# ============================================

display(
    spark.table(
        "workspace.default.target_orders"
    )
)

In [0]:
# ============================================
#  DROP TABLES
# ============================================

spark.sql("""
DROP TABLE IF EXISTS workspace.default.target_orders
""")

######JSON — Malformed Records
•	Use PERMISSIVE mode + _corrupt_record column in schema

•	Filter _corrupt_record IS NOT NULL → quarantine

•	Use try_cast() for safe type casting in merge set


In [0]:
################  Male formed Json ################


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer", StringType(), True),
    StructField("amount", StringType(), True) , # keep STRING initially for safety
      # REQUIRED
    StructField("_corrupt_record", StringType(), True)
])

json_df = (
    spark.read
    .format("json")
    .option("mode", "PERMISSIVE")   # do not fail pipeline
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(schema)
    .load("/Volumes/workspace/default/assignment/json/new_json - Copy.json")
)

display(json_df)

In [0]:
from pyspark.sql.functions import col, lit

bad_json_df = json_df.filter(
    col("_corrupt_record").isNotNull()
).withColumn(
    "error_reason",
    lit("Malformed JSON structure")
)

display(bad_json_df)

In [0]:
clean_df = json_df.filter(
    col("_corrupt_record").isNull()
).drop("_corrupt_record")

display(clean_df)

In [0]:
from pyspark.sql.functions import col,expr

clean_df = (
    clean_df
    .withColumn("order_id", col("order_id").cast("int"))
    .withColumn("amount", expr("try_cast(amount as int)")

))

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.target_orders (
    order_id INT,
    customer STRING,
    amount INT
)
""")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.quarantine_orders (
    order_id INTEGER,
    customer STRING,
    amount STRING,
    error_reason STRING
)
""")

In [0]:
bad_json_df.printSchema()
spark.table("workspace.default.quarantine_orders").printSchema()

In [0]:
bad_json_df.select(
    "order_id",
    "customer",
    "amount",
    "error_reason"
).write \
 .format("delta") \
 .mode("append") \
 .saveAsTable("workspace.default.quarantine_orders")

In [0]:
spark.table("workspace.default.target_orders").printSchema()
clean_df.printSchema()
# display(
#     clean_df
# )

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.default.target_orders"
)

(
    target.alias("t")

    .merge(
        clean_df.alias("s"),
        "t.order_id = s.order_id"
    )

    .whenMatchedUpdate(
        set={
            "customer": "s.customer",
            "amount": "try_cast(s.amount as int)"
        }
    )

    .whenNotMatchedInsert(
        values={
            "order_id": "s.order_id",
            "customer": "s.customer",
            "amount": "try_cast(s.amount as int)"
        }
    )

    .execute()
)

In [0]:
display(
    spark.table("workspace.default.target_orders")
)

In [0]:
# ============================================
#DROP TABLES
# ============================================

spark.sql("""
DROP TABLE IF EXISTS workspace.default.target_orders
""")

spark.sql("""
DROP TABLE IF EXISTS workspace.default.quarantine_orders
""")

######Mixed Sources
######<p>(CSV + JSON → same target)</p>

```
CSV source  →  standardise schema  ─┐
                                     ├── unionByName → dropDuplicates → MERGE
JSON source →  standardise schema  ─┘

⚠  All sources must match target schema before union. Cast types explicitly — never rely on inferSchema for production.


In [0]:
################  Reading and Merging both csv and json both from target to source  ################

In [0]:
# ============================================
# STEP 1 — CREATE TARGET TABLE
# ============================================

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.target_orders (
    order_id INT,
    customer STRING,
    amount INT
)
""")

In [0]:
csv_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .load("/Volumes/workspace/default/assignment/csv/csv_bad - Copy - Copy.csv")
)

display(csv_df)

In [0]:
json_df = (
    spark.read
    .format("json")
    .option("multiline", True)
    .load("/Volumes/workspace/default/assignment/json/new_json.json")
)

display(json_df)

In [0]:
from pyspark.sql.functions import col

csv_df = (
    csv_df

    .withColumn(
        "order_id",
        col("order_id").cast("int")
    )

    .withColumn(
        "amount",
        col("amount").cast("int")
    )

    .select(
        "order_id",
        "customer",
        "amount"
    )
)

In [0]:
# WHY STANDARDIZATION?

# Because different sources may have:

# different datatypes
# different column order
# different naming

# Before merge:
# all sources must match target schema.
json_df = (
    json_df

    .withColumn(
        "order_id",
        col("order_id").cast("int")
    )

    .withColumn(
        "amount",
        col("amount").cast("int")
    )

    .select(
        "order_id",
        "customer",
        "amount"
    )
)

In [0]:
csv_df.printSchema()
json_df.printSchema()

In [0]:
combined_df = csv_df.unionByName(json_df)

display(combined_df)

In [0]:
combined_df = combined_df.dropDuplicates(
    ["order_id"]
)

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.default.target_orders"
)

(
    target.alias("t")

    .merge(
        combined_df.alias("s"),
        "t.order_id = s.order_id"
    )

    .whenMatchedUpdate(
        set={
            "customer": "s.customer",
            "amount": "s.amount"
        }
    )

    .whenNotMatchedInsert(
        values={
            "order_id": "s.order_id",
            "customer": "s.customer",
            "amount": "s.amount"
        }
    )

    .execute()
)

In [0]:
display(
    spark.table(
        "workspace.default.target_orders"
    )
)

In [0]:
# ============================================
# DROP TABLES
# ============================================

spark.sql("""
DROP TABLE IF EXISTS workspace.default.target_orders
""")